In [1]:
import pandas as pd
import numpy as np 
from sqlalchemy import create_engine
import os 
import glob
import sweetviz as sv

In [14]:
# df = pd.read_csv('../Raw Data/olist_customers_dataset.csv')
# df2 = pd.read_csv('../Raw Data/olist_products_dataset.csv')
# print('file terbaca')
# df2.info()

In [2]:
def batch_cleaner(data_configuration, output ="Clean Data"):
    """
    Membersihkan banyak file sekaligus berdasarkan blueprint kolom masing-masing.
    """
    # Buat folder output jika belum ada
    os.makedirs(output , exist_ok=True)
    
    # Looping membaca konfigurasi tiap file
    for rules_name, spek in data_configuration.items():
        # Cari semua file yang sesuai dengan path (bisa pakai wildcard seperti *.csv)
        file_list = glob.glob(spek['path'])
        
        if not file_list:
            print(f"no files found in path: {spek['path']}")
            continue
            
        for file_path in file_list:
            file_name = os.path.basename(file_path)
            print(f"... Processing: {file_name} ({rules_name})...")
            
            try:
                # 1. Load Data
                df = pd.read_csv(file_path)
                
                # 2. Pembersihan Wajib untuk semua file
                df.columns = df.columns.str.strip()
                df = df.drop_duplicates()
                
                # 3. Pembersihan Spesifik berbasis parameter
                if 'str_column' in spek and spek['str_column']:
                    # Pastikan kolom ada di file sebelum dibersihkan
                    str_target = [c for c in spek['str_column'] if c in df.columns]
                    if str_target:
                        df[str_target] = df[str_target].apply(lambda x: x.astype(str).str.lower().str.strip())
                
                if 'datetime_column' in spek and spek['datetime_column']:
                    for col in spek['datetime_column']:
                        if col in df.columns:
                            df[col] = pd.to_datetime(df[col], errors='coerce')
                
                if 'critical/id_column' in spek and spek['critical/id_column']:
                    target_id = [c for c in spek['critical/id_column'] if c in df.columns]
                    if target_id:
                        df = df.dropna(subset=target_id)
                
                # 4. Ekspor Hasil Bersih
                path_output = os.path.join(output , f"cleaned_{file_name}")
                df.to_csv(path_output, index=False)
                print(f"saved on: {path_output}")
                
            except Exception as e:
                print(f"failed to process {file_name}. Error: {e}")
                
    print("\n Cleaning Done...")

In [3]:
data_configuration = {
    'Aturan_Pelanggan': {
        'path': '../Raw Data/olist_customers_dataset.csv', # Bisa diisi satu file spesifik
        'str_column': ['customer_city'],
        'critical/id_column': ['customer_id', 'customer_unique_id']
    },
    'aturan_orders': {
        'path': '../Raw Data/olist_order*.csv',
        'datetime_column': [
            'shipping_limit_date','review_creation_date', 'review_answer_timestamp',
            "order_approved_at","order_delivered_carrier_date","order_delivered_customer_date",
            "order_estimated_delivery_date"],
        'critical/id_column': [
            'order_id', 'product_id', 'order_item_id', 'seller_id',
            'review_id', 'review_score', 'customer_id', 'order_delivered_customer_date'
            ]
    },
    'aturan_products': {
        'path': '../Raw Data/olist_products_dataset.csv',
        'str_column': ['product_category_name'],
        'critical/id_column': ['product_id', "product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm"]
    },
    'aturan_sellers': {
        'path': '../Raw Data/olist_sellers_dataset.csv',
        'str_column': ['seller_city'],
        'critical/id_column': ['seller_id', 'seller_zip_code_prefix']
    }
}

In [4]:
batch_cleaner(data_configuration, output = 'clean data v.1')

... Processing: olist_customers_dataset.csv (Aturan_Pelanggan)...
saved on: clean data v.1\cleaned_olist_customers_dataset.csv
... Processing: olist_orders_dataset.csv (aturan_orders)...
saved on: clean data v.1\cleaned_olist_orders_dataset.csv
... Processing: olist_order_items_dataset.csv (aturan_orders)...
saved on: clean data v.1\cleaned_olist_order_items_dataset.csv
... Processing: olist_order_payments_dataset.csv (aturan_orders)...
saved on: clean data v.1\cleaned_olist_order_payments_dataset.csv
... Processing: olist_order_reviews_dataset.csv (aturan_orders)...
saved on: clean data v.1\cleaned_olist_order_reviews_dataset.csv
... Processing: olist_products_dataset.csv (aturan_products)...
saved on: clean data v.1\cleaned_olist_products_dataset.csv
... Processing: olist_sellers_dataset.csv (aturan_sellers)...
saved on: clean data v.1\cleaned_olist_sellers_dataset.csv

 Cleaning Done...


In [5]:
df_latih = pd.read_csv('../Raw Data/olist_customers_dataset.csv')
df_uji = pd.read_csv('../Notebook/clean data v.1/cleaned_olist_customers_dataset.csv')

# Membandingkan dua tabel yang berbeda
laporan_dua_tabel = sv.compare(
    [df_latih, "Raw Data"], 
    [df_uji, "Cleaned Data "]
)

laporan_dua_tabel.show_html('perbandingan raw and cleaned.html')

                                             |          | [  0%]   00:00 -> (? left)

Report perbandingan raw and cleaned.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [7]:
df_latih2 = pd.read_csv('../Raw Data/olist_orders_dataset.csv')
df_uji2 = pd.read_csv('../Notebook/clean data v.1/cleaned_olist_orders_dataset.csv')

# Membandingkan dua tabel yang berbeda
laporan_dua_tabel = sv.compare(
    [df_latih2, "Raw Data"], 
    [df_uji2, "Cleaned Data "]
)

laporan_dua_tabel.show_html('perbandingan raw and cleaned 2.html')

                                             |          | [  0%]   00:00 -> (? left)

Report perbandingan raw and cleaned 2.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
